# EDA des prix OMA

Analyse exploratoire des prix agricoles extraits d'un bulletin de l'Observatoire des Marchés Agricoles (OMA).


In [ ]:
from datetime import date
from pathlib import Path
import pandas as pd

from sini.parsers.oma import OmaPriceParser
from sini.scrapers.oma import OmaScraper


## 1. Chargement et préparation des données


In [ ]:
PDF_PATH = Path("../data/oma/communique_du_04_au_10_novembre_2021.pdf")
if not PDF_PATH.exists():
    raise FileNotFoundError(f"Bulletin introuvable : {PDF_PATH.resolve()}")

text = OmaScraper().extract_text(PDF_PATH.read_bytes())
parser = OmaPriceParser()
date_releve = date(2021, 11, 10)

records = []
records.extend(parser.parse_tableau_1(text=text, date_releve=date_releve))
records.extend(parser.parse_tableau_2(text=text, date_releve=date_releve))
records.extend(parser.parse_tableau_3(text=text, date_releve=date_releve))
records.extend(parser.parse_tableau_4(text=text, date_releve=date_releve))

df = pd.DataFrame([record.__dict__ for record in records])
df["date_releve"] = pd.to_datetime(df["date_releve"])
df["prix_kg"] = df["prix"]
mask = df["unite"].eq("100kg")
df.loc[mask, "prix_kg"] = df.loc[mask, "prix"] / 100

print("Nombre total de records :", len(df))


In [ ]:
df.head(10)


## 2. Structure et types des données


In [ ]:
print("Dimensions :", df.shape)
print("\nColonnes :", df.columns.tolist())
print("\nTypes de données :")
display(df.dtypes.to_frame("dtype"))


## 3. Valeurs manquantes


In [ ]:
missing = df.isna().sum().to_frame("nombre_manquant")
missing["pourcentage"] = (missing["nombre_manquant"] / len(df) * 100).round(2)
missing


Les valeurs manquantes concernent uniquement `variete`, ce qui est normal lorsque le bulletin ne précise pas de variété.


## 4. Doublons


In [ ]:
print("Doublons exacts :", df.duplicated().sum())

colonnes_uniques = ["date_releve", "type_prix", "culture", "variete", "marche"]
doublons_metier = df[df.duplicated(subset=colonnes_uniques, keep=False)]
print("Lignes concernées par les doublons métier :", len(doublons_metier))
doublons_metier


## 5. Prix invalides


In [ ]:
prix_invalides = df[df["prix"] <= 0]
print("Nombre de prix invalides :", len(prix_invalides))
prix_invalides


## 6. Répartition des données


In [ ]:
print("Répartition par type de prix :")
display(df["type_prix"].value_counts().to_frame("nombre"))
print("Répartition par unité :")
display(df["unite"].value_counts().to_frame("nombre"))


## 7. Statistiques descriptives


In [ ]:
df["prix_kg"].describe().round(2)


Les prix des grossistes exprimés pour 100 kg ont été convertis en prix par kg dans `prix_kg`.


## 8. Prix par culture et type


In [ ]:
analyse_culture = (
    df.groupby(["type_prix", "culture"])["prix_kg"]
    .agg(["count", "min", "mean", "max"])
    .round(2)
)
analyse_culture


## 9. Prix par marché


In [ ]:
analyse_marche = (
    df.groupby(["type_prix", "marche"])["prix_kg"]
    .agg(["count", "min", "mean", "max"])
    .round(2)
)
analyse_marche


## 10. Détection des valeurs aberrantes


In [ ]:
groupes = ["type_prix", "culture", "variete"]
statistiques = (
    df.groupby(groupes, dropna=False)["prix_kg"]
    .agg(["count", "mean", "std"])
    .rename(columns={"mean": "moyenne", "std": "ecart_type"})
)

df_analyse = df.merge(statistiques, on=groupes, how="left")
df_analyse["z_score"] = (
    (df_analyse["prix_kg"] - df_analyse["moyenne"])
    / df_analyse["ecart_type"]
)

valeurs_aberrantes = df_analyse[
    (df_analyse["count"] >= 3)
    & (df_analyse["z_score"].abs() > 2)
]

valeurs_aberrantes[
    ["type_prix", "culture", "variete", "marche",
     "prix_kg", "moyenne", "ecart_type", "z_score"]
]


## 11. Vérification de l'anomalie Maïs Pilé


In [ ]:
mais_pile = df[
    (df["culture"] == "Maïs") &
    (df["variete"] == "Pilé")
][
    ["date_releve", "type_prix", "marche", "prix_kg"]
].sort_values("prix_kg")

mais_pile


Le prix de **350 FCFA/kg à Kayes Centre** est statistiquement atypique, mais il correspond bien à la valeur du bulletin OMA. Il n'est donc pas corrigé.


## 12. Conclusion

L'analyse porte sur **134 observations**. Aucun doublon, prix invalide ou incohérence nécessitant une correction n'a été détecté. Les unités ont été normalisées en prix/kg et la valeur atypique du Maïs Pilé à Kayes a été vérifiée avec la source.
